# 알약 탐지·분류 통합 추론

학습된 탐지기와 분류기 `best.pt`를 불러와 원본 이미지에서 알약 위치와 세부 품목을 순서대로 예측한다. 두 모델의 학습은 기존 개별 노트북에서 수행하고, 이 노트북은 end-to-end 추론과 결과 저장에 사용한다.

In [ ]:
from importlib import import_module
from pathlib import Path
import json
import sys

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd

from classifier import (
    load_inference_model as load_classifier_model,
    validate_class_map,
)
from detector import (
    load_inference_model as load_detector_model,
    select_device,
    validate_detector_label_map,
    validate_inference_batch,
)
from pipeline import (
    plot_end_to_end_predictions,
    run_end_to_end_inference,
    save_end_to_end_artifacts,
)

In [ ]:
output_root = project_root / "outputs"
detector_run_name = "ssdlite320_baseline"
classifier_run_name = "resnet50_baseline"
inference_factory_path = ""  # 예: "data.detector_data:build_inference_loader"
inference_batch_size = 4
score_threshold = 0.25

detector_run_dir = output_root / detector_run_name
classifier_run_dir = output_root / classifier_run_name
detector_checkpoint_path = detector_run_dir / "best.pt"
classifier_checkpoint_path = classifier_run_dir / "best.pt"
detector_label_map_path = detector_run_dir / "label_map.json"
classifier_class_map_path = classifier_run_dir / "class_map.json"
result_dir = output_root / "end_to_end_inference"
device = select_device()
print(f"device={device}")

## 추론 데이터 연결

`inference_factory_path`가 가리키는 함수는 `factory(batch_size=inference_batch_size)` 호출을 지원하고 추론 DataLoader를 반환해야 한다. 각 batch는 `docs/detector_input_spec.md`의 추론 형식인 `images: list[Tensor]`와 `meta: list[dict]`를 제공한다. 이미지 tensor는 CPU의 RGB `float32 [3, H, W]`, 값 범위 `[0, 1]`이어야 한다.

In [ ]:
if not inference_factory_path:
    raise RuntimeError("inference_factory_path를 추론 DataLoader 구성 함수로 설정하세요")

module_name, function_name = inference_factory_path.split(":", maxsplit=1)
inference_factory = getattr(import_module(module_name), function_name)
detector_label_map = json.loads(detector_label_map_path.read_text(encoding="utf-8"))
class_map = json.loads(classifier_class_map_path.read_text(encoding="utf-8"))

validate_detector_label_map(detector_label_map)
validate_class_map(class_map, expected_num_classes=class_map["num_classes"])

inference_loader = inference_factory(batch_size=inference_batch_size)
validate_inference_batch(next(iter(inference_loader)))
# 첫 batch 확인으로 sampler 상태가 바뀌지 않도록 loader를 다시 만든다.
inference_loader = inference_factory(batch_size=inference_batch_size)
print("통합 추론 입력 검증 완료")

## Best checkpoint 로드

두 checkpoint는 현재 label map 및 class map과 일치해야 한다. 모델은 같은 CUDA, MPS 또는 CPU 장치에서 순서대로 실행된다.

In [ ]:
detector_model, device = load_detector_model(
    detector_checkpoint_path,
    detector_label_map,
    device=device,
)
classifier_model, device = load_classifier_model(
    classifier_checkpoint_path,
    class_map,
    expected_num_classes=class_map["num_classes"],
    device=device,
)
print("탐지기와 분류기 checkpoint 로드 완료")

## End-to-end 추론

탐지 bbox를 이미지 경계로 보정한 뒤 crop한다. 가로세로 비율을 유지하기 위한 여백은 ImageNet RGB 평균에 해당하는 `(124, 116, 104)`로 채운다. 따라서 ImageNet 정규화 후 padding 영역은 거의 0이 되어 검정이나 흰색 여백보다 중립적인 입력이 된다. 최종 점수는 `detector_score × classifier_score`로 계산한다.

In [ ]:
result = run_end_to_end_inference(
    detector_model,
    classifier_model,
    inference_loader,
    class_map,
    device,
    score_threshold=score_threshold,
    detector_name=detector_run_name,
    show_progress=True,
)
predictions_path, errors_path = save_end_to_end_artifacts(result, result_dir)
print(f"predictions={len(result.predictions)}, detector_errors={len(result.detector_errors)}, classifier_errors={len(result.classifier_errors)}")
print(f"saved: {predictions_path}")
print(f"saved: {errors_path}")

In [ ]:
prediction_table = pd.DataFrame(
    [prediction.to_dict() for prediction in result.predictions]
)
display(prediction_table.head(20))

error_table = pd.DataFrame(
    [
        {"stage": "detector", **error.to_dict()}
        for error in result.detector_errors
    ]
    + [
        {"stage": "classifier", **error.to_dict()}
        for error in result.classifier_errors
    ]
)
display(error_table.head(20))

## 예측 시각화

첫 번째 추론 이미지에 최종 품목명, bbox와 결합 점수를 표시하고 PNG로 저장한다.

In [ ]:
visualization_loader = inference_factory(batch_size=inference_batch_size)
first_batch = next(iter(visualization_loader))
first_image = first_batch["images"][0]
first_image_id = str(first_batch["meta"][0]["image_id"])
first_predictions = [
    prediction
    for prediction in result.predictions
    if prediction.image_id == first_image_id
]
figure = plot_end_to_end_predictions(
    first_image,
    first_predictions,
    title=f"End-to-end image_id={first_image_id}",
)
visualization_dir = result_dir / "visualizations"
visualization_dir.mkdir(parents=True, exist_ok=True)
figure.savefig(
    visualization_dir / f"image-{first_image_id}.png",
    bbox_inches="tight",
)
figure